In [1]:
import tkinter as tk
import random
import time

# ---------------- 설정 ----------------
MIN_DELAY_MS = 1200   # 랜덤 대기 최소
MAX_DELAY_MS = 3500   # 랜덤 대기 최대

# ---------------- 상태 ----------------
state = "idle"        # idle -> waiting -> go
start_time = None
after_id = None
best_ms = None

def set_ui(bg, title, sub):
    root.configure(bg=bg)
    title_var.set(title)
    sub_var.set(sub)

def reset():
    global state, start_time, after_id
    state = "idle"
    start_time = None
    if after_id is not None:
        root.after_cancel(after_id)
    set_ui("#0f172a", "반응속도 테스트", "Start를 누르고, 초록으로 바뀌면 바로 클릭!")
    btn_start.config(state="normal")
    btn_reset.config(state="disabled")

def start_test():
    global state, after_id
    state = "waiting"
    btn_start.config(state="disabled")
    btn_reset.config(state="normal")
    set_ui("#7f1d1d", "준비...", "초록으로 바뀌면 클릭! (지금 누르면 부정출발)")
    delay = random.randint(MIN_DELAY_MS, MAX_DELAY_MS)
    after_id = root.after(delay, go_signal)

def go_signal():
    global state, start_time
    state = "go"
    start_time = time.perf_counter()
    set_ui("#065f46", "지금!!!", "바로 클릭하세요!")

def on_click(event=None):
    global state, best_ms
    if state == "idle":
        return

    if state == "waiting":
        # 부정출발
        set_ui("#1f2937", "부정출발 😵", "너무 빨랐음! Reset 후 다시!")
        state = "idle"
        btn_start.config(state="disabled")
        return

    if state == "go":
        elapsed = (time.perf_counter() - start_time) * 1000
        ms = int(elapsed)

        if best_ms is None or ms < best_ms:
            best_ms = ms
            best_var.set(f"최고 기록: {best_ms} ms 🏆")
        else:
            best_var.set(f"최고 기록: {best_ms} ms")

        set_ui("#0f172a", f"{ms} ms", "Start로 다시 도전!")
        state = "idle"
        btn_start.config(state="normal")

# ---------------- UI ----------------
root = tk.Tk()
root.title("⚡ 반응속도 테스트")
root.geometry("520x320")
root.resizable(False, False)

title_var = tk.StringVar()
sub_var = tk.StringVar()
best_var = tk.StringVar(value="최고 기록: -")

title = tk.Label(root, textvariable=title_var, font=("맑은 고딕", 26, "bold"), fg="white", bg="#0f172a")
title.pack(pady=25)

sub = tk.Label(root, textvariable=sub_var, font=("맑은 고딕", 12), fg="#cbd5e1", bg="#0f172a")
sub.pack()

best = tk.Label(root, textvariable=best_var, font=("맑은 고딕", 12, "bold"), fg="#facc15", bg="#0f172a")
best.pack(pady=12)

btn_frame = tk.Frame(root, bg="#0f172a")
btn_frame.pack(pady=18)

btn_start = tk.Button(
    btn_frame, text="Start",
    command=start_test,
    font=("맑은 고딕", 14, "bold"),
    bg="#38bdf8", fg="#0f172a",
    activebackground="#0ea5e9",
    bd=0, width=10, cursor="hand2"
)
btn_start.pack(side="left", padx=10)

btn_reset = tk.Button(
    btn_frame, text="Reset",
    command=reset,
    font=("맑은 고딕", 14, "bold"),
    bg="#1e293b", fg="white",
    activebackground="#334155",
    bd=0, width=10, cursor="hand2",
    state="disabled"
)
btn_reset.pack(side="left", padx=10)

# 화면 어디든 클릭하면 판정
root.bind("<Button-1>", on_click)

reset()
root.mainloop()